In [1]:
import re
import shutil

import nibabel as nib
import numpy as np
import colorcet as cc
import pandas as pd

import matplotlib.pyplot as plt

from pathlib import Path
from tqdm import tqdm
from time import time

from nilearn.image import math_img
from nipype.interfaces.ants import ApplyTransforms, Registration
from nipype.interfaces.fsl import FLIRT, FNIRT, ApplyWarp, InvWarp
from nipype.interfaces import afni
from scipy.ndimage import binary_fill_holes

# Reportlab for pdf 

from reportlab.pdfbase import pdfmetrics
from reportlab.lib.utils import ImageReader
from reportlab.pdfgen import canvas

/mnt/8TB/JC/anaconda3/envs/JCmri/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


260901-16:57:39,299 nipype.utils WARNING:
	 A newer version (1.11.0) of nipy/nipype is available. You are using 1.8.6


# Helper functions

In [6]:

############################################################################################################# 

############################################# ANTS registration #############################################

############################################################################################################# 

def compute_registration_ants(
    file_name, 
    ref_name, 
    output_dir, 
    registration_type = 'SyN', 
    prefix= 'file_name',
    write_composite_transform = True,
    num_threads = 18,
):
    
    '''
    Performs registration using ants registration from nipype module

    file_name: (str)
    registration: (str)             Type of registration (either: Rigid, Affine or Syn) - performs a composition of them 
    '''
    
    reg = Registration()

    reg.inputs.fixed_image = str(ref_name)
    reg.inputs.moving_image = str(file_name)
    reg.inputs.dimension = 3
    reg.inputs.initial_moving_transform_com = 1
    reg.inputs.write_composite_transform = write_composite_transform
    reg.inputs.collapse_output_transforms = False
    reg.inputs.num_threads = num_threads
    
    if registration_type == 'SyN':
        
        reg.inputs.output_transform_prefix = str(output_dir / f'{prefix}_')
        reg.inputs.transforms = ['Rigid', 'Affine', 'SyN']
        reg.inputs.transform_parameters = [(0.1,), (0.1,), (0.25, 3.0, 1.0)]
        reg.inputs.number_of_iterations = [[1000,500,250,100], [1500, 200], [100, 50, 30]]
        reg.inputs.smoothing_sigmas = [[3,2,1,0], [1,0], [4,2,1]]
        reg.inputs.shrink_factors = [[8,4,2,1], [2,1], [8,4,2]]
        reg.inputs.metric = ['MI', 'MI', 'MI']
        reg.inputs.radius_or_number_of_bins = [32, 32, 32]
        reg.inputs.metric_weight = [1]*3 # Default (value ignored currently by ANTs)
    
    elif registration_type == 'Affine':

        reg.inputs.output_transform_prefix = str(output_dir / f'{prefix}_')
        reg.inputs.transforms = ['Rigid', 'Affine']
        reg.inputs.transform_parameters = [(0.1,), (0.1,)]
        reg.inputs.number_of_iterations = [[1000,500,250,100], [1500, 200]]
        reg.inputs.smoothing_sigmas = [[3,2,1,0], [1,0]]
        reg.inputs.shrink_factors = [[8,4,2,1], [2,1]]
        reg.inputs.metric = ['MI', 'MI']
        reg.inputs.radius_or_number_of_bins = [32, 32]
        reg.inputs.metric_weight = [1]*2 # Default (value ignored currently by ANTs)
        
    elif registration_type == 'Rigid':
        
        reg.inputs.output_transform_prefix = str(output_dir / f'{prefix}_')
        reg.inputs.transforms = ['Rigid']
        reg.inputs.transform_parameters = [(0.1,)]
        reg.inputs.number_of_iterations = [[1000,500,250,100]]
        reg.inputs.smoothing_sigmas = [[3,2,1,0]]
        reg.inputs.shrink_factors = [[8,4,2,1]]
        reg.inputs.metric = ['MI']
        reg.inputs.radius_or_number_of_bins = [32]
        reg.inputs.metric_weight = [1] # Default (value ignored currently by ANTs)

    print(reg.cmdline)
    
    reg.run()
    
    return output_dir / f'{prefix}_Composite.h5'

def apply_transformation_ants(
    file_name, 
    ref_name, 
    transform_path,
    output_dir,
    prefix = 'registered_image',
    interpolation = 'Linear',
    invert_transform_flag = [False]
):
    
    """
    Apply a transformation computed by ANTsRegistration
    """
    
    at = ApplyTransforms()

    at.inputs.dimension = 3
    at.inputs.input_image = str(file_name)
    at.inputs.reference_image = str(ref_name)
    at.inputs.output_image = str(output_dir / f'{prefix}.nii.gz')
    at.inputs.interpolation = interpolation
    at.inputs.default_value = 0
    at.inputs.transforms = str(transform_path)
    at.inputs.invert_transform_flags = invert_transform_flag

    at.cmdline
    
    at.run()
    
    return str(output_dir / f'{prefix}.nii.gz')

############################################################################################################ 

############################################# FLIRT registration ###########################################

############################################################################################################

def compute_transform_flirt(
    file_name,
    ref_name,
    save_dir
):

    flt = FLIRT()
    flt.inputs.in_file = str(file_name)
    flt.inputs.reference = str(ref_name)
    
    flt.inputs.output_type = "NIFTI_GZ"
    flt.inputs.out_file = str(save_dir / (file_name.name.split('_pwd')[0] + '_proc-registeredFLIRT_pwd.nii.gz')) 
    flt.inputs.out_matrix_file = str(save_dir / (file_name.name.split('_pwd')[0] + '_registrationMatrix.mat'))
    print(flt.cmdline)
    res = flt.run()

    return save_dir / (file_name.name.split('_pwd')[0] + '_registrationMatrix.mat')

def apply_transform_flirt(
    file_name,
    ref_name,
    transformation_path,
    save_dir,
    prefix = None,
    interp = 'spline'
):

    if prefix is None:
        perfix = (file_name.name.split('_pwd')[0] + '_proc-registeredFLIRT_pwd')
    
    flt = FLIRT()
    flt.inputs.in_file = str(file_name)
    flt.inputs.reference = str(ref_name)

    flt.inputs.apply_xfm = True
    flt.inputs.in_matrix_file = transformation_path
    flt.inputs.output_type = "NIFTI_GZ"
    flt.inputs.interp = interp
    flt.inputs.out_file = str(save_dir / f'{prefix}.nii.gz') 
    print(flt.cmdline)
    res = flt.run()

    return f'{prefix}.nii.gz'

############################################################################################################ 

############################################# FNIRT registration ###########################################

############################################################################################################

def compute_transform_fnirt(
    file_blownup_name,
    template_blownup_name,
    save_dir,
    prefix,
    affine_initialisation,
    refmask = None,
    inmask = None
):
    
    fnirt_mprage = fsl.FNIRT()
    fnirt_mprage.inputs.affine_file=affine_initialisation
    fnirt_mprage.inputs.in_file=file_blownup_name
    fnirt_mprage.inputs.ref_file=template_blownup_name
    fnirt_mprage.inputs.warped_file=str(save_dir / (prefix + '.nii.gz'))
    fnirt_mprage.inputs.in_fwhm = [8, 4, 2, 2]
    fnirt_mprage.inputs.subsampling_scheme = [4, 2, 1, 1]
    fnirt_mprage.inputs.regularization_lambda = [200, 75, 40, 20]
    if not (refmask is None):
        fnirt_mprage.inputs.refmask = refmask
    if not (inmask is None):
        fnirt_mprage.inputs.inmask = inmask
    
    fnirt_mprage.inputs.warp_resolution = (5,5,5)
    
    fnirt_mprage.cmdline

    fnirt_mprage.run()

    return save_dir / (prefix + '.nii.gz'), save_dir / (file_blownup_name.name.split('.nii.gz')[0] + '_warpcoef.nii.gz') 

def apply_transform_fnirt(
    file_blownup_name,
    template_blownup_name,
    transformation_path,
    save_dir,
    prefix,
    interp = 'trilinear'
):
    
    aw = fsl.ApplyWarp()
    aw.inputs.in_file = str(file_blownup_name)
    aw.inputs.ref_file = str(template_blownup_name)
    aw.inputs.field_file = str(transformation_path)
    aw.inputs.out_file = str(save_dir / (prefix + '.nii.gz'))
    aw.inputs.interp = interp

    aw.cmdline

    aw.run()

    return save_dir / (prefix + '.nii.gz')

def inverse_transform_fnirt(
    transformation_path,
    ref_img
):

    invwarp = InvWarp()
    invwarp.inputs.warp = transformation_path
    invwarp.inputs.reference = ref_img
    invwarp.inputs.output_type = "NIFTI_GZ"
    invwarp.inputs.inverse_warp = str(transformation_path.parent / (transformation_path.name.split('.nii.gz')[0] + 'Inv.nii.gz'))
    
    invwarp.cmdline

    invwarp.run() 

    return transformation_path.parent / (transformation_path.name.split('.nii.gz')[0] + 'Inv.nii.gz')

############################################################################################################# 

############################################### AFNI 3dAllineate ###########################################

############################################################################################################# 

def compute_registration_afni(
    file_name,
    template_name,
    save_dir,
    prefix
):

    allineate = afni.Allineate()
    allineate.inputs.in_file = file_name
    allineate.inputs.reference = template_name
    allineate.inputs.master = template_name
    allineate.inputs.out_file = str(save_dir / f'{prefix}.nii.gz') 
    allineate.inputs.autoweight = "+10"
    allineate.inputs.center_of_mass = ""
    allineate.inputs.out_param_file = str(save_dir / f'{prefix.split("pwd")[0] + "warp.param.1D"}') 
    allineate.inputs.args = "-cost lpa"
    
    allineate.cmdline
    
    allineate.run()

    return str(save_dir / f'{prefix}.nii.gz'), str(save_dir / f'{prefix.split("pwd")[0] + "warp.param.1D"}') 

def apply_transform_afni(
    file_name,
    template_name,
    save_dir,
    prefix,
    param_name,
    interp = 'linear'
):

    allineate = afni.Allineate()
    allineate.inputs.in_file = file_name
    allineate.inputs.out_file = template_name
    allineate.inputs.master = template_name
    allineate.inputs.in_param_file = param_name
    allineate.inputs.out_file = str(save_dir / f'{prefix}.nii.gz') 
    allineate.inputs.interpolation = interp
    
    allineate.cmdline
    
    allineate.run()
    
    return str(save_dir / f'{prefix}.nii.gz') 

############################################################################################################# 

################################################## AFNI Qwarp ###############################################

############################################################################################################# 

def compute_transform_qwarp(
    file_name,
    template_name,
    save_dir,
    prefix
):
    
    qwarp = afni.Qwarp()
    qwarp.inputs.in_file = file_name
    qwarp.inputs.base_file = template_name
    qwarp.inputs.out_file = str(save_dir / f'{prefix}.nii.gz') 
    qwarp.inputs.resample = True
    qwarp.inputs.lpa = True
    qwarp.inputs.iwarp = True
    qwarp.inputs.allineate = True
    qwarp.inputs.allineate_opts = "-com -cost lpa -autoweight +10"
    qwarp.inputs.blur = [0,-3]
    qwarp.inputs.minpatch = 10
    qwarp.inputs.num_threads = 18
    qwarp.inputs.quiet = True  
    
    qwarp.cmdline

    qwarp.run()
    
    return save_dir / f'{prefix}.nii.gz'


def apply_transform_qwarp(
    file_name,
    template_name,
    save_dir,
    prefix,
    transformation_path,
    interp = 'tri-linear'
):
    
    nwarp = afni.NwarpApply()
    nwarp.inputs.in_file = file_name
    nwarp.inputs.master = template_name
    nwarp.inputs.warp = str(transformation_path)
    nwarp.inputs.out_file = save_dir / f'{prefix}.nii.gz'
    nwarp.inputs.interp = interp
    nwarp.inputs.quiet = True
    nwarp.cmdline
    
    res = nwarp.run() 
    
    return save_dir / f'{prefix}.nii.gz'

############################################################################################################# 

################################################ Other tools ###############################################

############################################################################################################# 

def clean_file(
    file_name,
    output_dir, 
    temp_dir
):

    target_folder = output_dir / file_name.parent.parent.name / file_name.parent.name 
    target_folder.mkdir(exist_ok = True, parents = True)

    for content_name in temp_dir.iterdir():

        target_name = target_folder / content_name.name
        shutil.move(content_name, target_name)
    
    return temp_dir

def display_overlay(
    img,
    bg_img,
    slice_ids = np.linspace(5,75,16).astype(int),
    nb_col = 4,
    vmin = 0.8,
    vmax = 2.5,
    flag_mask = False,
    title = None
):

    nb_slices = len(slice_ids)
    nb_lin = int(nb_slices/nb_col)

    fig, ax  = plt.subplots(nb_lin, nb_col, figsize = (nb_col*6, nb_lin*5), dpi = 200, facecolor = 'k')

    for i,slice_id in enumerate(slice_ids):

        img_2_display = np.log10(bg_img.get_fdata()[slice_id,:,:])
        img_2_display[np.isnan(img_2_display)] = 0
        img_2_display[np.isinf(img_2_display)] = 0
        ax[i//nb_col, i%nb_col].imshow(img_2_display, cmap = 'Greys_r', vmin = vmin, vmax = vmax, aspect = 0.1/0.11)
        ax[i//nb_col, i%nb_col].axis('off')

        if flag_mask:

            img_2_display = (img.get_fdata()[slice_id,:,:])
            ax[i//nb_col, i%nb_col].imshow(img_2_display, cmap = cc.cm['fire'], vmin = 0, vmax = 2, aspect = 0.1/0.11, alpha = 0.5*(img_2_display>0))
            ax[i//nb_col, i%nb_col].axis('off')            
    
        else:
            
            img_2_display = np.log10(img.get_fdata()[slice_id,:,:])
            ax[i//nb_col, i%nb_col].imshow(img_2_display, cmap = cc.cm['fire'], vmin = 0.6, vmax = 2.2, aspect = 0.1/0.11, alpha = 0.5*(img_2_display>0))
            ax[i//nb_col, i%nb_col].axis('off')

    if not title is None:

        fig.suptitle(title, c = 'w', fontsize = 30)

    plt.tight_layout()

    return fig


# General parameters

In [14]:
# Fixed image
ref_name = Path('../1_data/templates/source-BI_space-fUSI_desc-GillianTemplate_res-110umx100umx100um_feature.nii.gz')

# Moving image
file_name = Path('../1_data/example_data/sub-cp230515a/ses-CPXISO/angio/sub-cp230515a_ses-CPXISO_acq-3dscan_probe-linear_pwd.nii.gz')

# ANTs

## Affine

In [16]:
output_dir = Path('/mnt/8TB/JC/3_Trash/2026-09-01_test-fUSIBrain2026-Registration/ANTs-affine/')
output_dir.mkdir(exist_ok = True, parents = True)

transform_path = compute_registration_ants(
    file_name, 
    ref_name, 
    output_dir, 
    registration_type = 'Affine', 
    prefix= f'{file_name.name.split(".nii.gz")[0]}_ANTs-Aff',
    write_composite_transform = True,
    num_threads = 18,
)

# Apply the transform to the image

registered_name = apply_transformation_ants(
    file_name, 
    ref_name, 
    transform_path,
    output_dir,
    prefix = f'{file_name.name.split(".nii.gz")[0]}_registered-ANTs-Aff',
    interpolation = 'Linear'
)


antsRegistration --collapse-output-transforms 0 --dimensionality 3 --initial-moving-transform [ ../1_data/templates/source-BI_space-fUSI_desc-GillianTemplate_res-110umx100umx100um_feature.nii.gz, ../1_data/example_data/sub-cp230515a/ses-CPXISO/angio/sub-cp230515a_ses-CPXISO_acq-3dscan_probe-linear_pwd.nii.gz, 1 ] --initialize-transforms-per-stage 0 --interpolation Linear --output /mnt/8TB/JC/3_Trash/2026-09-01_test-fUSIBrain2026-Registration/ANTs-affine/sub-cp230515a_ses-CPXISO_acq-3dscan_probe-linear_pwd_ANTs-Aff_ --transform Rigid[ 0.1 ] --metric MI[ ../1_data/templates/source-BI_space-fUSI_desc-GillianTemplate_res-110umx100umx100um_feature.nii.gz, ../1_data/example_data/sub-cp230515a/ses-CPXISO/angio/sub-cp230515a_ses-CPXISO_acq-3dscan_probe-linear_pwd.nii.gz, 1, 32 ] --convergence [ 1000x500x250x100, 1e-06, 10 ] --smoothing-sigmas 3.0x2.0x1.0x0.0 --shrink-factors 8x4x2x1 --use-histogram-matching 1 --transform Affine[ 0.1 ] --metric MI[ ../1_data/templates/source-BI_space-fUSI_desc-

## SyN

In [13]:
output_dir = Path('/mnt/8TB/JC/3_Trash/2026-09-01_test-fUSIBrain2026-Registration/ANTs-SyN/')
output_dir.mkdir(exist_ok = True, parents = True)

transform_path = compute_registration_ants(
    file_name, 
    ref_name, 
    output_dir, 
    registration_type = 'SyN', 
    prefix= f'{file_name.name.split(".nii.gz")[0]}_ANTs-SyN',
    write_composite_transform = True,
    num_threads = 18,
)

# Apply the transform to the image

registered_name = apply_transformation_ants(
    file_name, 
    ref_name, 
    transform_path,
    output_dir,
    prefix = f'{file_name.name.split(".nii.gz")[0]}_registered-ANTs-SyN',
    interpolation = 'Linear'
)

antsRegistration --collapse-output-transforms 0 --dimensionality 3 --initial-moving-transform [ ../1_data/templates/source-BI_space-fUSI_desc-GillianTemplate_res-110umx100umx100um_feature.nii.gz, ../1_data/example_data/sub-cp230515a/ses-CPXISO/angio/sub-cp230515a_ses-CPXISO_acq-3dscan_probe-linear_pwd.nii.gz, 1 ] --initialize-transforms-per-stage 0 --interpolation Linear --output /mnt/8TB/JC/3_Trash/2026-09-01_test-fUSIBrain2026-Registration/ANTs-SyN/sub-cp230515a_ses-CPXISO_acq-3dscan_probe-linear_pwd_ANTs-SyN_ --transform Rigid[ 0.1 ] --metric MI[ ../1_data/templates/source-BI_space-fUSI_desc-GillianTemplate_res-110umx100umx100um_feature.nii.gz, ../1_data/example_data/sub-cp230515a/ses-CPXISO/angio/sub-cp230515a_ses-CPXISO_acq-3dscan_probe-linear_pwd.nii.gz, 1, 32 ] --convergence [ 1000x500x250x100, 1e-06, 10 ] --smoothing-sigmas 3.0x2.0x1.0x0.0 --shrink-factors 8x4x2x1 --use-histogram-matching 1 --transform Affine[ 0.1 ] --metric MI[ ../1_data/templates/source-BI_space-fUSI_desc-Gil

# FSL

## FLIRT

In [17]:
output_dir = Path('/mnt/8TB/JC/3_Trash/2026-09-01_test-fUSIBrain2026-Registration/FLIRT/')
output_dir.mkdir(exist_ok = True, parents = True)

transform_path = compute_transform_flirt(
    file_name,
    ref_name,
    output_dir
)

# Apply the transform to the image

registered_name = apply_transform_flirt(
    file_name,
    ref_name,
    transform_path,
    output_dir,
    prefix = f'{file_name.name.split(".nii.gz")[0]}_FLIRT',
    interp = 'spline'
)

flirt -in ../1_data/example_data/sub-cp230515a/ses-CPXISO/angio/sub-cp230515a_ses-CPXISO_acq-3dscan_probe-linear_pwd.nii.gz -ref ../1_data/templates/source-BI_space-fUSI_desc-GillianTemplate_res-110umx100umx100um_feature.nii.gz -out /mnt/8TB/JC/3_Trash/2026-09-01_test-fUSIBrain2026-Registration/FLIRT/sub-cp230515a_ses-CPXISO_acq-3dscan_probe-linear_proc-registeredFLIRT_pwd.nii.gz -omat /mnt/8TB/JC/3_Trash/2026-09-01_test-fUSIBrain2026-Registration/FLIRT/sub-cp230515a_ses-CPXISO_acq-3dscan_probe-linear_registrationMatrix.mat
flirt -in ../1_data/example_data/sub-cp230515a/ses-CPXISO/angio/sub-cp230515a_ses-CPXISO_acq-3dscan_probe-linear_pwd.nii.gz -ref ../1_data/templates/source-BI_space-fUSI_desc-GillianTemplate_res-110umx100umx100um_feature.nii.gz -out /mnt/8TB/JC/3_Trash/2026-09-01_test-fUSIBrain2026-Registration/FLIRT/sub-cp230515a_ses-CPXISO_acq-3dscan_probe-linear_pwd_FLIRT.nii.gz -omat sub-cp230515a_ses-CPXISO_acq-3dscan_probe-linear_pwd_flirt.mat -applyxfm -init /mnt/8TB/JC/3_Tra

# AFNI

## Affine - 3dAllineate

In [22]:
output_dir = Path('/mnt/8TB/JC/3_Trash/2026-09-01_test-fUSIBrain2026-Registration/3dAllineate/')
output_dir.mkdir(exist_ok = True, parents = True)

local_file_name, param_path = compute_registration_afni(
        file_name,
        ref_name,
        output_dir,
        prefix=f'{file_name.name.split(".nii.gz")[0]}_registered-3dAllineate'
    )

260901-17:42:17,176 nipype.interface INFO:
	 stderr 2026-09-01T17:42:17.175982:++ 3dAllineate: AFNI version=AFNI_25.1.08 (May  6 2025) [64-bit]
260901-17:42:17,178 nipype.interface INFO:
	 stderr 2026-09-01T17:42:17.175982:++ Authored by: Zhark the Registrator
260901-17:42:17,180 nipype.interface INFO:
	 stderr 2026-09-01T17:42:17.180094:++ Option '-cmass' enables center-of-mass code = 7 = +xyz
260901-17:42:17,181 nipype.interface INFO:
	 stderr 2026-09-01T17:42:17.180094:** AFNI converts NIFTI_datatype=64 (FLOAT64) in file /home/jmariani/Documents/python/libraries/fUSIBrain2026_registration-workshop/1_data/example_data/sub-cp230515a/ses-CPXISO/angio/sub-cp230515a_ses-CPXISO_acq-3dscan_probe-linear_pwd.nii.gz to FLOAT32
260901-17:42:17,182 nipype.interface INFO:
	 stderr 2026-09-01T17:42:17.180094:     Warnings of this type will be muted for this session.
260901-17:42:17,183 nipype.interface INFO:
	 stderr 2026-09-01T17:42:17.180094:     Set AFNI_NIFTI_TYPE_WARN to YES to see them all,